!['Resting state networks seminal paper](../imgs/Beckmann_2005_title.png)

https://pubmed.ncbi.nlm.nih.gov/16087444/

## Importing

In [ ]:
import nilearn as nil
import nibabel as nib
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from nilearn import plotting, surface, datasets, input_data, decomposition
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm # For having progressbar during loops
from nitime.timeseries import TimeSeries
from nitime.analysis import FilterAnalyzer
from sklearn.decomposition import PCA
from nilearn.connectome import ConnectivityMeasure
from matplotlib import colors
from matplotlib import patches

# Some utils

### Signal filter

In [ ]:
def signal_filter(nifti_img, mask, standardize=False, ub=0.08, lb=0.008):
    print('Filtering the signal...')
    img = nifti_img.get_fdata()
    img_affine = nifti_img.affine
    TR = nifti_img.header.get_zooms()[3]
    
    n_vox_mask = np.sum(mask==1) # Number of voxels in the mask
    mask_cord = np.where(mask==1)
    img_filtered = np.zeros(img.shape)
#     print(img_filtered.shape)
    for v in tqdm(range(n_vox_mask)):
        voxel_ts = img[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] # time series of the single voxel
        T = TimeSeries(voxel_ts.T, sampling_interval=TR)
        F = FilterAnalyzer(T, ub=0.08, lb=0.008) #0.08, 0.008
        tmp_TC_filt = F.filtered_boxcar.data
        tmp_TC_filt = tmp_TC_filt.T
        if standardize:
            tmp_TC_filt = (tmp_TC_filt-np.mean(tmp_TC_filt))/np.std(tmp_TC_filt)
            
        # Check if there are nan values
        if np.sum(np.isnan(tmp_TC_filt))!=0:
            for t in tmp_TC_filt:
                tmp_TC_filt[t] = 0
        img_filtered[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] = tmp_TC_filt

    return nib.Nifti1Image(img_filtered, img_affine)

### Custom masker and inverse_masker

In [ ]:
def custom_masker(nifti_img, nifti_mask):
    mask = nifti_mask.get_fdata()
    img = nifti_img.get_fdata()
    voxels_ts = img[np.where(mask==1)]
    return voxels_ts

def custom_inv_masker(voxel_ts, nifti_mask):
    mask = nifti_mask.get_fdata()
    mask_affine = nifti_mask.affine
    img = np.zeros(mask.shape)
    mask_coord = np.where(mask==1)
    for v_i, v in enumerate(voxel_ts):
        img[mask_coord[0][v_i], mask_coord[1][v_i], mask_coord[2][v_i]] = v
    nifti_img = nib.Nifti1Image(img, mask_affine)
    return nifti_img

# Load an example subject

In [ ]:
# main_path='E:/CorsoFisica/rest_STD/' # Path to the folder with the subjects
# main_path='C:/Unito/PhD/CorsoFisica/data/rest_STD/' # Path to the folder with the subjects - without denoising
main_path='C:/Unito/PhD/CorsoFisica/data/rest_STD_clean/' # Path to the folder with the subjects - with denoising

filelist = os.listdir(main_path) # List of the subjects

sub_ = nib.load(main_path+filelist[0]) # Load the nii.gz file of the subject
sub_header = sub_.header # Subject's header
sub_affine = sub_.affine # Subject's affine
sub = sub_.get_fdata() # Get the numpy version of the nii.gz file
print(sub.shape) # x*y*z*time

TR = sub_.header.get_zooms()[3]

# Load the brain mask

In [ ]:
# main_path='E:/CorsoFisica/group_mask/' # Path to the folder with the subjects
main_path='C:/Unito/PhD/CorsoFisica/data/group_mask/' # Path to the folder with the subjects
filename='group_mask.nii.gz' # List of the subjects

group_mask_ = nib.load(main_path+filename) # Load the nii.gz file of the subject
group_mask_header = group_mask_.header # Subject's header
group_mask_affine = group_mask_.affine # Subject's affine
group_mask = group_mask_.get_fdata() # Get the numpy version of the nii.gz file
print(group_mask.shape) # x*y*z*time
n_vox_mask = np.sum(group_mask==1) # Number of voxels within the mask

# Load the atlas

In [ ]:
which_atlas = 'glasser' # Chose the atlas
main_path='C:/Unito/PhD/CorsoFisica/data/atlas/' # Path to the folder with the atlas
filename = 'glasser_MNI152NLin6Asym_labels_p20_resamp.nii.gz'
atlas_labels_file = 'glasser_labels.csv'
delimiter = ','

atlas_ = nib.load(main_path+'/'+which_atlas+'/'+filename) # Load the nii.gz file of the subject
atlas_header = atlas_.header # Subject's header
atlas_affine = atlas_.affine # Subject's affine
atlas = atlas_.get_fdata() # Get the numpy version of the nii.gz file
print(atlas.shape) # x*y*z*time

# Labels
atlas_labels = pd.read_csv(main_path+'/'+which_atlas+'/'+atlas_labels_file, delimiter=delimiter)
print(atlas.min(), atlas.max())

In [ ]:
plotting.plot_roi(atlas_, title=which_atlas+' atlas', display_mode='ortho')

In [ ]:
# Subdivision in cortices of the glasser atlas
network_list = atlas_labels['cortex'].unique().tolist()
# network_list

# Apply the PCA on a single subject

## Filter the signal

### Filtering with nilern masker

In [ ]:
%%time
masker = input_data.NiftiMasker(mask_img=atlas_mask_, low_pass=0.08, high_pass=0.008, t_r=TR, standardize=True) # To apply the brain mask to our subject
voxels_ts = masker.fit_transform(sub_).T
plt.plot(voxels_ts[564])

## PCA

In [ ]:
# Apply PCA to the time series data
n_components = 10  # Adjust the number of components based on your needs
pca = PCA(n_components=n_components)
pca_components = pca.fit_transform(voxels_ts.T)

# Explained variance ratio (to see how much variance each component explains)
explained_variance = pca.explained_variance_ratio_
print(f'Explained Variance Ratio: {explained_variance}')

# Inverse transform PCA components to project them back to the brain image space
# pca_brain_maps = masker.inverse_transform(pca.components_)
pca_brain_maps = []
for component in pca.components_:
    nifti_component = custom_inv_masker(component, atlas_mask_)
    pca_brain_maps.append(nifti_component)

# Plot the first component as a brain map
plotting.plot_stat_map(pca_brain_maps[0], title='PCA Component 1',
                       threshold=0.01)

# Show the plot
plotting.show()

In [ ]:
thresh = 0.00
for i in range(n_components):
    nii_img = pca_brain_maps[i]
#     plotting.plot_stat_map(nii_img, title='PCA Component '+str(i),
#                        threshold=0.01)
    
    fsaverage = datasets.fetch_surf_fsaverage()
    texture_left = surface.vol_to_surf(nii_img, fsaverage.pial_left)
    texture_left[np.where(texture_left==0)] = np.nan 
    texture_right = surface.vol_to_surf(nii_img, fsaverage.pial_right)
    texture_right[np.where(texture_right==0)] = np.nan 

    fig, axs = plt.subplots(2,2, subplot_kw={'projection': '3d'}, figsize=(4,4))
    # Left hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                           title='Left', view='lateral',colorbar=False, threshold=thresh, axes=axs[0][0])
    # Right hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                           title='Right', view='lateral', colorbar=False, threshold=thresh, axes=axs[0][1])
    # Right hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                                view='medial', colorbar=False, threshold=thresh, axes=axs[1][0])
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                                view='medial', colorbar=False, threshold=thresh, axes=axs[1][1])
    fig.suptitle('Component '+str(i+1), x=0.51, y=1)
    plt.tight_layout()
    plotting.show()


# Try the ICA

The nilearn implementation of the ICA is inspired by the following paper:
https://pubmed.ncbi.nlm.nih.gov/20153834/

In [ ]:
%%time
n_components = 25
ica = decomposition.CanICA(n_components=n_components, mask=atlas_mask_,
                           low_pass=0.08, high_pass=0.008, t_r=TR,standardize=True)

ica.fit(sub_)

In [ ]:
components_img = ica.components_img_
# Loop through components and plot each on the surface (choose one or more components)
thresh = 0
for i in range(n_components):
    component = components_img.get_fdata()[:,:,:,i]
    component = nib.Nifti1Image(component, atlas_affine)
    # Project the ith ICA component to the cortical surface (left hemisphere)
    texture_left = surface.vol_to_surf(component, fsaverage.pial_left)
    texture_left[np.where(texture_left==0)] = np.nan
    # Project the ith ICA component to the cortical surface (right hemisphere)
    texture_right = surface.vol_to_surf(component, fsaverage.pial_right)
    texture_right[np.where(texture_right==0)] = np.nan
    
    fig, axs = plt.subplots(2,2, subplot_kw={'projection': '3d'}, figsize=(4,4))
    # Left hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                           title='Left', view='lateral',colorbar=False, threshold=thresh, axes=axs[0][0])
    # Right hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                           title='Right', view='lateral', colorbar=False, threshold=thresh, axes=axs[0][1])
    # Right hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                                view='medial', colorbar=False, threshold=thresh, axes=axs[1][0])
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                                view='medial', colorbar=False, threshold=thresh, axes=axs[1][1])
    fig.suptitle('Component '+str(i+1), x=0.51, y=1)
    plt.tight_layout()
    plotting.show()

    
# Show the plots
plotting.show()

# Multisubjects analysis
Let's concatenate all the experimental subjects and do PCA and ICA

In [ ]:
# main_path='C:/Unito/PhD/CorsoFisica/data/rest_STD/' # Path to the folder with the subjects
main_path='C:/Unito/PhD/CorsoFisica/data/rest_STD_clean/' # Path to the folder with the subjects
filelist = os.listdir(main_path) # List of the subjects
all_sub_concat = np.empty((sub.shape[0], sub.shape[1], sub.shape[2],0))
for subfile in filelist:# Loop over subjects

    sub_ = nib.load(main_path+subfile) # Load the nii.gz file of the subject
    sub_header = sub_.header # Subject's header
    sub_affine = sub_.affine # Subject's affine
    sub = sub_.get_fdata() # Get the numpy version of the nii.gz file
#     print(sub.shape) # x*y*z*time
    TR = sub_.header.get_zooms()[3]
    all_sub_concat = np.concatenate((all_sub_concat, sub), axis=-1)
    print(all_sub_concat.shape)
    
nifti_all_sub_concat = nib.Nifti1Image(all_sub_concat, sub_affine)


In [ ]:
# Just put all the subjects in a list
# main_path='C:/Unito/PhD/CorsoFisica/data/rest_STD/' # Path to the folder with the subjects
main_path='C:/Unito/PhD/CorsoFisica/data/rest_STD_clean/' # Path to the folder with the subjects
filelist = os.listdir(main_path) # List of the subjects
nifti_all_sub_list = []
for subfile in filelist:# Loop over subjects
    sub_ = nib.load(main_path+subfile) # Load the nii.gz file of the subject
    nifti_all_sub_list.append(sub_)


## PCA

In [ ]:
%%time
masker = input_data.NiftiMasker(mask_img=atlas_mask_, low_pass=0.08, high_pass=0.008, t_r=TR, standardize=True) # To apply the brain mask to our subject
voxels_ts_all_sub = masker.fit_transform(nifti_all_sub_concat).T
plt.plot(voxels_ts_all_sub[1044])

In [ ]:
# Apply PCA to the time series data
n_components = 25  # Adjust the number of components based on your needs
pca = PCA(n_components=n_components)
pca_components = pca.fit_transform(voxels_ts_all_sub.T)

# Explained variance ratio (to see how much variance each component explains)
explained_variance = pca.explained_variance_ratio_
print(f'Explained Variance Ratio: {explained_variance}')

# Inverse transform PCA components to project them back to the brain image space
# pca_brain_maps = masker.inverse_transform(pca.components_)
pca_brain_maps = []
for component in pca.components_:
    nifti_component = custom_inv_masker(component, atlas_mask_)
    pca_brain_maps.append(nifti_component)

# Plot the first component as a brain map
plotting.plot_stat_map(pca_brain_maps[0], title='PCA Component 1',
                       threshold=0.01)

# Show the plot
plotting.show()

In [ ]:
thresh = 0.00
for i in range(n_components)[:10]:
    nii_img = pca_brain_maps[i]
#     plotting.plot_stat_map(nii_img, title='PCA Component '+str(i),
#                        threshold=0.01)
    
    fsaverage = datasets.fetch_surf_fsaverage()
    texture_left = surface.vol_to_surf(nii_img, fsaverage.pial_left)
    texture_right = surface.vol_to_surf(nii_img, fsaverage.pial_right)

    fig, axs = plt.subplots(2,2, subplot_kw={'projection': '3d'}, figsize=(4,4))
    # Left hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                           title='Left', view='lateral',colorbar=False, threshold=thresh, axes=axs[0][0])
    # Right hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                           title='Right', view='lateral', colorbar=False, threshold=thresh, axes=axs[0][1])
    # Right hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                                view='medial', colorbar=False, threshold=thresh, axes=axs[1][0])
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                                view='medial', colorbar=False, threshold=thresh, axes=axs[1][1])
    fig.suptitle('Component '+str(i+1), x=0.51, y=1)
    plt.tight_layout()
    plotting.show()

    
# Show the plots
plotting.show()


## ICA

In [ ]:
%%time
n_components = 25
ica = decomposition.CanICA(n_components=n_components, mask=atlas_mask_,
                           low_pass=0.08, high_pass=0.008, t_r=TR,standardize=True)

ica.fit(nifti_all_sub_list)

In [ ]:
components_img = ica.components_img_
# Loop through components and plot each on the surface (choose one or more components)
thresh = 0.00
for i in range(n_components):
    component = components_img.get_fdata()[:,:,:,i]
    component = nib.Nifti1Image(component, atlas_affine)
    # Project the ith ICA component to the cortical surface (left hemisphere)
    texture_left = surface.vol_to_surf(component, fsaverage.pial_left)
    texture_left[np.where(texture_left==0)] = np.nan    
    # Project the ith ICA component to the cortical surface (right hemisphere)
    texture_right = surface.vol_to_surf(component, fsaverage.pial_right)
    texture_right[np.where(texture_right==0)] = np.nan
    
    fig, axs = plt.subplots(2,2, subplot_kw={'projection': '3d'}, figsize=(4,4))
    # Left hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                           title='Left', view='lateral',colorbar=False, threshold=thresh, axes=axs[0][0])
    # Right hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                           title='Right', view='lateral', colorbar=False, threshold=thresh, axes=axs[0][1])
    # Right hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                                view='medial', colorbar=False, threshold=thresh, axes=axs[1][0])
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                                view='medial', colorbar=False, threshold=thresh, axes=axs[1][1])
    fig.suptitle('Component '+str(i+1), x=0.51, y=1)
    plt.tight_layout()
    plotting.show()

    
# Show the plots
plotting.show()

# Multisubject-ICA with noisy data

# PCA (and ICA) over regions space

### Extract the time series for each region of the brain using an atlas

In [ ]:
sub_filtered_ = signal_filter(sub_, atlas_mask)
sub_filtered = sub_filtered_.get_fdata()
sub_parc = np.empty((0,sub.shape[-1])) # parcellized brain
for a_idx in range(1, int(atlas.max())+1): # Loop over areas
    area_coord = np.where(atlas==a_idx) # coordinates of the current brain region
    area_ts = sub[area_coord].mean(axis=0) # time series of the current region
    sub_parc = np.concatenate((sub_parc, area_ts[np.newaxis]), axis=0)
print(sub_parc.shape)

## Region-based PCA Single subject

In [ ]:
# Apply PCA to the time series data
n_components = 10  # Adjust the number of components based on your needs
pca = PCA(n_components=n_components)
pca_components = pca.fit_transform(sub_parc.T)

# Explained variance ratio (to see how much variance each component explains)
explained_variance = pca.explained_variance_ratio_
print(f'Explained Variance Ratio: {explained_variance}')

# Inverse transform PCA components to project them back to the brain image space
# pca_brain_maps = masker.inverse_transform(pca.components_)
pca_brain_maps = []
for component in pca.components_:
    nifti_component = custom_inv_masker(component, atlas_mask_)
    pca_brain_maps.append(nifti_component)

In [ ]:
# Put the PCA components in the brain space
pca_components = np.zeros((len(pca.components_),sub.shape[0],sub.shape[1],sub.shape[2]))
for c_i, component in enumerate(pca.components_):
    for a_idx in range(1, int(atlas.max())+1): # Loop over areas
        area_coord = np.where(atlas==a_idx) # coordinates of the current brain region
        pca_components[c_i][area_coord] = component[a_idx-1]

In [ ]:
pca_components.shape

In [ ]:
thresh = 0.00
for i in range(n_components):
    nii_img = nib.Nifti1Image(pca_components[i], atlas_affine)
#     plotting.plot_stat_map(nii_img, title='PCA Component '+str(i),
#                        threshold=0.01)
    
    fsaverage = datasets.fetch_surf_fsaverage()
    texture_left = surface.vol_to_surf(nii_img, fsaverage.pial_left)
    texture_left[np.where(texture_left==0)] = np.nan 
    texture_right = surface.vol_to_surf(nii_img, fsaverage.pial_right)
    texture_right[np.where(texture_right==0)] = np.nan 

    fig, axs = plt.subplots(2,2, subplot_kw={'projection': '3d'}, figsize=(4,4))
    # Left hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                           title='Left', view='lateral',colorbar=False, threshold=thresh, axes=axs[0][0])
    # Right hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                           title='Right', view='lateral', colorbar=False, threshold=thresh, axes=axs[0][1])
    # Right hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                                view='medial', colorbar=False, threshold=thresh, axes=axs[1][0])
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                                view='medial', colorbar=False, threshold=thresh, axes=axs[1][1])
    fig.suptitle('Component '+str(i+1), x=0.51, y=1)
    plt.tight_layout()
    plotting.show()

    
# Show the plots
plotting.show()

## Region-based PCA Multi subject

In [ ]:
main_path='C:/Unito/PhD/CorsoFisica/data/rest_STD_clean/' # Path to the folder with the subjects
all_parc = np.empty((int(atlas.max()), 0))
for sub_file in filelist:
    sub_ = nib.load(main_path+sub_file)
    sub_filtered_ = signal_filter(sub_, atlas_mask)
    sub_filtered = sub_filtered_.get_fdata()
    sub_parc = np.empty((0,sub.shape[-1])) # parcellized brain
    for a_idx in range(1, int(atlas.max())+1): # Loop over areas
        area_coord = np.where(atlas==a_idx) # coordinates of the current brain region
        area_ts = sub_filtered[area_coord].mean(axis=0) # time series of the current region
        sub_parc = np.concatenate((sub_parc, area_ts[np.newaxis]), axis=0)
#     print(sub_parc.shape)
    all_parc = np.concatenate((all_parc,sub_parc), axis=1)
print(all_parc.shape)


In [ ]:
# Apply PCA to the time series data
n_components = 10  # Adjust the number of components based on your needs
pca = PCA(n_components=n_components)
pca_components = pca.fit_transform(all_parc.T)

# Explained variance ratio (to see how much variance each component explains)
explained_variance = pca.explained_variance_ratio_
print(f'Explained Variance Ratio: {explained_variance}')

# Inverse transform PCA components to project them back to the brain image space
# pca_brain_maps = masker.inverse_transform(pca.components_)
pca_brain_maps = []
for component in pca.components_:
    nifti_component = custom_inv_masker(component, atlas_mask_)
    pca_brain_maps.append(nifti_component)

In [ ]:
# Put the PCA components in the brain space
pca_components = np.zeros((len(pca.components_),sub.shape[0],sub.shape[1],sub.shape[2]))
for c_i, component in enumerate(pca.components_):
    for a_idx in range(1, int(atlas.max())+1): # Loop over areas
        area_coord = np.where(atlas==a_idx) # coordinates of the current brain region
        pca_components[c_i][area_coord] = component[a_idx-1]

In [ ]:
thresh = 0.00
for i in range(n_components):
    nii_img = nib.Nifti1Image(pca_components[i], atlas_affine)
#     plotting.plot_stat_map(nii_img, title='PCA Component '+str(i),
#                        threshold=0.01)
    
    fsaverage = datasets.fetch_surf_fsaverage()
    texture_left = surface.vol_to_surf(nii_img, fsaverage.pial_left)
    texture_left[np.where(texture_left==0)] = np.nan 
    texture_right = surface.vol_to_surf(nii_img, fsaverage.pial_right)
    texture_right[np.where(texture_right==0)] = np.nan 

    fig, axs = plt.subplots(2,2, subplot_kw={'projection': '3d'}, figsize=(4,4))
    # Left hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                           title='Left', view='lateral',colorbar=False, threshold=thresh, axes=axs[0][0])
    # Right hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                           title='Right', view='lateral', colorbar=False, threshold=thresh, axes=axs[0][1])
    # Right hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                                view='medial', colorbar=False, threshold=thresh, axes=axs[1][0])
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                                view='medial', colorbar=False, threshold=thresh, axes=axs[1][1])
    fig.suptitle('Component '+str(i+1), x=0.51, y=1)
    plt.tight_layout()
    plotting.show()

    
# Show the plots
plotting.show()

# PCA over whole brain correlation matrix

## Single subject

In [ ]:
# Extract the time series for each region of the brain using an atlas

In [ ]:
sub_filtered_ = signal_filter(sub_, atlas_mask)
sub_filtered = sub_filtered_.get_fdata()
sub_parc = np.empty((0,sub.shape[-1])) # parcellized brain
for a_idx in range(1, int(atlas.max())+1): # Loop over areas
    area_coord = np.where(atlas==a_idx) # coordinates of the current brain region
    area_ts = sub[area_coord].mean(axis=0) # time series of the current region
    sub_parc = np.concatenate((sub_parc, area_ts[np.newaxis]), axis=0)
print(sub_parc.shape)

In [ ]:
sub_parc.shape

In [ ]:
correlation_measure = ConnectivityMeasure(kind='correlation')
correlation_matrix = correlation_measure.fit_transform([sub_parc.T])
print(correlation_matrix.shape)

In [ ]:
# Setting to make a cool plot of the connectivity matrix
lobe_list = atlas_labels['Lobe'].unique()
lobe_sizes = []  # Number of regions in each group (left and right)
for hemi in ['L','R']:
    sub_atlas = atlas_labels.loc[atlas_labels['LR']==hemi]
    for lobe in lobe_list:
        size = len(sub_atlas.loc[sub_atlas['Lobe']==lobe])
        lobe_sizes.append(size)
print(lobe_sizes)
        
group_colors = ['red', 'blue', 'green', 'purple', 'gray']*2  # Colors for the group labels

# Cumulative sum of lobe sizes
cumsum_sizes = np.cumsum([0] + lobe_sizes)

In [ ]:
# Create a custom color bar for the groups
fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.imshow(correlation_matrix[0], vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks([])
ax.set_yticks([])

# Color bar for the matrix
plt.colorbar(cax, ax=ax)
plt.xlim(-20, )
plt.ylim(-20, 360)

# Add dividing lines for each group
for size in cumsum_sizes:
    ax.axhline(size - 0.5, color='gray', linewidth=1)
    ax.axvline(size - 0.5, color='gray', linewidth=1)

# Add colored rectangles for each group - x axis
for idx, color in enumerate(group_colors):
    rect = patches.Rectangle((-20, cumsum_sizes[idx] - 0.5), 10, lobe_sizes[idx], linewidth=0.1, edgecolor='gray', facecolor=color)
    ax.add_patch(rect)
    
# Add colored rectangles for each group - y axis
for idx, color in enumerate(group_colors):
    rect = patches.Rectangle((cumsum_sizes[idx] - 0.5, -20), lobe_sizes[idx], 10, linewidth=0.1, edgecolor='gray', facecolor=color)
    ax.add_patch(rect)

# Annotate group names
for idx, label in enumerate(lobe_list.tolist()*2):
    plt.text(-25, (cumsum_sizes[idx] + cumsum_sizes[idx + 1]) / 2, label, fontsize=10, va='center', 
             ha='right', color=group_colors[idx])

plt.title('Single subject Correlation Matrix')
plt.show()

### PCA with the correlation matrix

In [ ]:
# Apply PCA to the time series data
n_components = 10  # Adjust the number of components based on your needs
pca = PCA(n_components=n_components)
pca_components = pca.fit_transform(correlation_matrix[0])

# Explained variance ratio (to see how much variance each component explains)
explained_variance = pca.explained_variance_ratio_
print(f'Explained Variance Ratio: {explained_variance}')

# Inverse transform PCA components to project them back to the brain image space
# pca_brain_maps = masker.inverse_transform(pca.components_)
pca_brain_maps = []
for component in pca.components_:
    nifti_component = custom_inv_masker(component, atlas_mask_)
    pca_brain_maps.append(nifti_component)

In [ ]:
# Put the PCA components in the brain space
pca_components = np.zeros((len(pca.components_),sub.shape[0],sub.shape[1],sub.shape[2]))
for c_i, component in enumerate(pca.components_):
    for a_idx in range(1, int(atlas.max())+1): # Loop over areas
        area_coord = np.where(atlas==a_idx) # coordinates of the current brain region
        pca_components[c_i][area_coord] = component[a_idx-1]

In [ ]:
thresh = 0.00
for i in range(n_components):
    nii_img = nib.Nifti1Image(pca_components[i], atlas_affine)
#     plotting.plot_stat_map(nii_img, title='PCA Component '+str(i),
#                        threshold=0.01)
    
    fsaverage = datasets.fetch_surf_fsaverage()
    texture_left = surface.vol_to_surf(nii_img, fsaverage.pial_left)
    texture_left[np.where(texture_left==0)] = np.nan 
    texture_right = surface.vol_to_surf(nii_img, fsaverage.pial_right)
    texture_right[np.where(texture_right==0)] = np.nan 

    fig, axs = plt.subplots(2,2, subplot_kw={'projection': '3d'}, figsize=(4,4))
    # Left hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                           title='Left', view='lateral',colorbar=False, threshold=thresh, axes=axs[0][0])
    # Right hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                           title='Right', view='lateral', colorbar=False, threshold=thresh, axes=axs[0][1])
    # Right hemisphere
    plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                                view='medial', colorbar=False, threshold=thresh, axes=axs[1][0])
    plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                                view='medial', colorbar=False, threshold=thresh, axes=axs[1][1])
    fig.suptitle('Component '+str(i+1), x=0.51, y=1)
    plt.tight_layout()
    plotting.show()

    

## Multi subject